THIS NOTEBOOK WAS WRITEN TO IMPLMENT QMIX FOR MULTI AGENT REINFORCEMENT LEARNING 

In [1]:
import os 
import numpy as np

import torch 
import torch.nn as nn
import torch.nn.functional as F 

import copy
import torch.optim as optim

In [2]:
# Add ENV SIMULATION
import yaml
from src.env.warehouse_env import WarehouseEnv

# 1. Load the warehouse configuration (e.g., grid size, agents)
with open("configs/env_config.yaml") as f:
    config = yaml.safe_load(f)

# 2. Instantiate the environment
env = WarehouseEnv(config)

# 3. Pull the dynamic dimensions straight from the environment!
# (You can replace your hardcoded N_AGENTS, N_ACTIONS variables with these)
N_AGENTS = env.n_agents
N_ACTIONS = env.action_dim
N_OBS_DIM = env.obs_dim

# Recompute N_FEATURES 
# (You were using N_GRID before, but now you dynamically grab the exact observation size per agent)
N_FEATURES = N_AGENTS + N_OBS_DIM + N_ACTIONS
#N_FEATURES =  N_OBS_DIM + 1

In [3]:
# N_AGENTS = 2 
N_GRIDE = 10*10
#N_ACTIONS = 5
#N_FEATURES = N_AGENTS+N_GRRIDE+N_ACTION #input_shape
HIDDEN_DIM = 32
MIXER_HIDDERN_DIM = 32

In [4]:
# RNN for Q 
# Function GET_AGENT_Q(observation, prev_action, hidden_state):
#     input = Concatenate(observation, prev_action)
#     new_hidden_state = GRU_Cell(input, hidden_state)
#     q_values = Linear_Layer(new_hidden_state)
#     Return q_values, new_hidden_state
# deal with pytorch shape(N_AGENT, N_AGENTS, N_FEATURES)

class QMixAgent(nn.Module):
    def __init__(self, input_shape, hidden_dim, n_actions):
        super(QMixAgent, self).__init__()
        self.hidden_dim = hidden_dim

        # Feature Extraction (MLP)
        self.fc1 = nn.Linear(input_shape, hidden_dim)

        # Reccurrent Layer (GRU)
        self.rnn = nn.GRUCell(hidden_dim, hidden_dim)

        # Q-value Output
        self.fc2 = nn.Linear(hidden_dim, n_actions)

    def init_hidden(self, batch_size = 1):
        return torch.zeros(batch_size, self.hidden_dim)

    def forward(self, inputs, hidden_state):
        """
        inputs: [Batch * N_AGENTS, N_FEATURES]
        hidden_state: [Batch * N_AGENTS, hidden_dim]
        """
        # Process raw observations
        x = F.relu(self.fc1(inputs))

        # Update the memory (GRU)
        h_in = hidden_state.reshape(-1, self.hidden_dim)
        h_out = self.rnn(x, h_in)

        # Get Q-values for all actions 
        q_values = self.fc2(h_out)

        return q_values, h_out

In [5]:
# Create the agent network 
agent_network = QMixAgent(N_FEATURES, HIDDEN_DIM, N_ACTIONS)

def get_agent_Q(agent_id, observation, prev_action, hidden_state): 

    agent_id_one_hot = np.eye(N_AGENTS)[agent_id]

    action_one_hot = np.eye(N_ACTIONS)[int(prev_action)]

    combined_input = np.concatenate([agent_id_one_hot, observation, action_one_hot])
    input_tensor = torch.tensor(combined_input, dtype = torch.float32).unsqueeze(0)

    q_values, new_hidden_state = agent_network(input_tensor, hidden_state)

    return q_values, new_hidden_state

In [6]:
# Mixed Network 
class QMixer(nn.Module):
    def __init__(self, n_agents, state_dim, embed_dim, hypernet_embed_dim):
        super(QMixer, self).__init__()

        self.n_agents = n_agents
        self.state_dim = state_dim
        self.embed_dim = embed_dim

        
        # --- Layer 1 hypernetworks ---
        self.hyper_w_1 = nn.Sequential(
            nn.Linear(state_dim, hypernet_embed_dim),
            nn.ReLU(),
            nn.Linear(hypernet_embed_dim, embed_dim * n_agents)
        )

        self.hyper_b_1 = nn.Linear(state_dim, embed_dim)
         
        # --- HyperNet for Layer 2 Weights ---

        self.hyper_w_2 = nn.Sequential(
            nn.Linear(state_dim, hypernet_embed_dim), 
            nn.ReLU(),
            nn.Linear(hypernet_embed_dim, embed_dim)
        )

        self.hyper_b_2 = nn.Sequential(
            nn.Linear(state_dim, hypernet_embed_dim), 
            nn.ReLU(),
            nn.Linear(embed_dim, 1)
        )

    
    def forward(self, agent_qs, state):
        # agent_qs shape: [batch_size, n_agents]
        # state shape: [batch_size, state_dim]
        batch_size = agent_qs.size(0)

        q_values = agent_qs.view(batch_size, 1, self.n_agents)

        w1 = torch.abs(self.hyper_w_1(state))
        w1 = w1.view(batch_size, self.n_agents, self.embed_dim)
        b1 = self.hyper_b_1(state).view(batch_size, 1, self.embed_dim)

        hidden = F.elu(torch.bmm(q_values, w1)+b1)

        # Generate Layer 2 weights from the global state
        w2 = self.hyper_w_2(state)

        # Enforce monotnicity (weights must stretly non-negative)
        w2 = torch.abs(w2).view(-1, self.embed_dim, 1)

        # Generate baises 
        b2 = self.hyper_b_2(state).view(-1,1,1)

        # Apply Layer 2 dynamically via batched matrix multiplication (BMM)
        # Note: 'hidden' comes from Layer 1, shape [batch_size, 1, embed_dim]
        q_tot = torch.bmm(hidden, w2) + b2 

        q_tot = q_tot.view(batch_size, -1)
        
        return q_tot
    

In [7]:
# ε-greedy Algo
# Function SELECT_ACTION(agent_q_values, epsilon):
#     If Random() < epsilon:
#         Return Random_Action()
#     Else:
#         Return ArgMax(agent_q_values)

def epsilon_greedy(q, epsilon = 0.1):
        if np.random.random() < epsilon:
            return np.random.randint(0,5)
        else:
            return int(np.argmax(q))

In [8]:
def get_global_state(env_observations):
    # Simply flatten all robot observations into one giant 1D array
    return np.concatenate(env_observations)

In [10]:
# QMIX implementation
NUM_EPISODES = 20 
MAX_STEPS = 5
GAMMA = 0.9
BATCH_SIZE = 32
TARGET_UPDATE_INTERNAL = 20
epsilon = 0.1
# actions = [[0,0], [0,1],[0,-1],[-1,0],[1,0]] # drop, up, down, left, right

D = [ ]

agent_network = QMixAgent(N_FEATURES, HIDDEN_DIM, N_ACTIONS)
mixer_network = QMixer(N_AGENTS, state_dim = (N_OBS_DIM*N_AGENTS), embed_dim = 64, hypernet_embed_dim=64)


target_agent = copy.deepcopy(agent_network)
target_mixer = copy.deepcopy(mixer_network)

all_parameters = list(agent_network.parameters())+list(mixer_network.parameters())
optimizer = optim.RMSprop(all_parameters, lr = 0.005)

for episode in range(NUM_EPISODES) :

	observations = env.reset()
	global_state = get_global_state(observations)

	hidden_state = agent_network.init_hidden(batch_size = N_AGENTS)
	previous_actions = np.zeros(N_AGENTS)

	for t in range(MAX_STEPS):

		current_actions = []
		
		# --- DECENTRALIZED EXECTUTION ---
		for i in range(N_AGENTS):
			obs_i = observations[i]
			prev_act_i = previous_actions[i]

			h_i = hidden_state[i].unsqueeze(0)

			# Get Q-values for all 5 actions
			q_values, new_h_i = get_agent_Q(i, obs_i, prev_act_i, h_i)
			
			action_i = epsilon_greedy(q_values.detach().numpy(), epsilon)
			current_actions.append(action_i) 

			hidden_state[i] = new_h_i.squeeze(0)

		# --- ENVIRONMENT STEP ----  
		next_observations, rewards, dones, info = env.step(current_actions)
		next_global_state = get_global_state(next_observations)

		# --- STORE EXPERIENCE ----
		transitions = {
			"obs": observations, 
			"actions": current_actions,
			"rewards": rewards,
			"next_obs": next_observations, 
			"state": global_state,
			"next_state": next_global_state,
			"dones": dones,
			"prev_actions": list(previous_actions)
		}
		D.append(transitions) 

		observations = next_observations
		global_state = next_global_state
		prev_actions = current_actions

	# --- CENTRALIZED TRAINING ---
	if len(D) >= BATCH_SIZE:

		# 1. Sample minibatch randomly from D
		batch_indices = np.random.choice(len(D), BATCH_SIZE, replace = False)
		batch = [D[idx] for idx in batch_indices]

		b_obs = torch.tensor([b['obs'] for b in batch], dtype = torch.float32)
		b_actions = torch.tensor([b['actions'] for b in batch], dtype = torch.int64).unsqueeze(-1)
		b_rewards = torch.tensor([b['rewards'] for b in batch], dtype = torch.float32).sum(dim=1, keepdim = True)
		b_next_obs = torch.tensor([b['next_obs'] for b in batch], dtype=torch.float32)
		b_states = torch.tensor([b['state'] for b in batch], dtype=torch.float32)
		b_next_states = torch.tensor([b['next_state'] for b in batch], dtype=torch.float32)
    
		# Check if ANY agent is done to mark the team as done
		b_dones = torch.tensor([any(b['dones']) for b in batch], dtype=torch.float32).unsqueeze(-1)

		# 2. For each agent i, compute Q_i (τ_i, a_i)
		h_zeros = agent_network.init_hidden(BATCH_SIZE * N_AGENTS)

		# Make agent IDS for the whole patch 
		b_agent_ids = torch.arange(N_AGENTS).unsqueeze(0).expand(BATCH_SIZE, -1)
		b_agent_id_oh = F.one_hot(b_agent_ids, num_classes=N_AGENTS).float()

		# Restructure ONLINE iputs (State = 71 + ID = 2 + Prev Action = 5 -> TOTAL = 78)
		b_prev_act = torch.tensor(np.array([b['prev_actions'] for b in batch]), dtype=torch.int64)
		b_prev_act_oh = F.one_hot(b_prev_act, num_classes=N_ACTIONS).float()
		b_inputs = torch.cat([b_agent_id_oh, b_obs, b_prev_act_oh], dim=-1)
		flat_obs = b_inputs.view(-1, N_FEATURES)

		# Resctructure TRAGET inputs for next states 
		b_act_oh = F.one_hot(b_actions.squeeze(-1), num_classes=N_ACTIONS).float()
		b_next_inputs = torch.cat([b_agent_id_oh, b_next_obs, b_act_oh], dim = -1)
		flat_next_obs = b_next_inputs.view(-1, N_FEATURES)

		# Forward Pass Online Agent
		all_q_vals, _ = agent_network(flat_obs, h_zeros)
		all_q_vals = all_q_vals.view(BATCH_SIZE, N_AGENTS, N_ACTIONS)

		# Exact only the Q-values of the actions that were actually token 
		choose_action_q_vals = torch.gather(all_q_vals, dim = 2, index = b_actions).squeeze(2)

		# Forward Pass Target Agent for the next state
		target_q_vals, _ = target_agent(flat_next_obs, h_zeros)
		target_q_vals = target_q_vals.view(BATCH_SIZE, N_AGENTS, N_ACTIONS)

		target_max_q_vals = target_q_vals.max(dim=2)[0]

		# 3. Mix Individual Q-values into Total Q-values Compute Q_tot(τ, a) = MixingNetwork(Q_i, s)
		q_tot = mixer_network(choose_action_q_vals, b_states)
		target_q_tot = target_mixer(target_max_q_vals, b_next_states)

		# 4. Compute Target Q_tot’ = r + γ max_a' Q_tot'(τ', a'; θ^-)
		expected_q_tot = b_rewards + GAMMA *(1-b_dones) * target_q_tot

		# 5. Update networks: Minimize TD Loss using gradient descent 
		loss = F.mse_loss(q_tot, expected_q_tot.detach()) # Detach target to prevent gradients flowing backwards

		optimizer.zero_grad()
		loss.backward()

		torch.nn.utils.clip_grad_norm_(all_parameters, 10.0)

		optimizer.step()

	# 6. Periodically update target networks
	if episode % TARGET_UPDATE_INTERNAL == 0:
		target_agent.load_state_dict(agent_network.state_dict())
		target_mixer.load_state_dict(mixer_network.state_dict())

/Users/jlhuang12/.pyenv/versions/3.11.4/envs/reinforcement_learning/lib/python3.11/site-packages/gymnasium/utils/passive_env_checker.py:245: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'list'>
  logger.warn(
/var/folders/sq/88x0d5tj7lv3vrbd2ch54ywc0000gn/T/ipykernel_14270/1989694707.py:77: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:278.)
  b_obs = torch.tensor([b['obs'] for b in batch], dtype = torch.float32)
